# Functions & Methods

## What's covered

- `def` — the method form, the workhorse you'll use ninety percent of the time
- The function-value form — `(x: Int) => x + 1` — what it actually creates and when you reach for it
- The real difference between *methods* and *function values* on the java virtual machine, and the `eta-expansion` that quietly bridges them
- Higher-order functions — passing functions as arguments, returning them as results
- Lambdas, the underscore shorthand, and when each is more readable
- Named arguments and default parameters — the two features that kill positional-argument bugs
- Multiple parameter lists (curried form) — and what they enable: type inference, partial application, and the receiver-style ay-pee-eyes you see in Spark
- Currying with `.curried` and the manual form, and partial application with `_`
- By-name parameters (`=> A`) — passing the *expression*, not its value, for control-flow ay-pee-eyes
- Vararg parameters
- The `Function0`, `Function1`, ... `Function22` family — what a function value actually IS
- Methods inside methods (local functions)
- The `inline` keyword preview — when you want a `def` to disappear at compile time

## `def` — the method form

The everyday way to declare a piece of executable code in Scala is `def`. It defines a *method* — a member of a class, trait, or singleton object. Top-level `def` (no enclosing object) compiles to a static method behind the scenes.

The full shape, with everything optional that is optional:

```
def methodName[TypeParams](param1: Type1, param2: Type2): ReturnType =
  body
```

Two things to remember from the start:

- **The return type can be inferred, but for public methods you should write it.** The notebook 02 rule for `val` applies here even more strongly. The return type is part of the method's contract.
- **The last expression in the body is the return value.** No `return` keyword needed. (You *can* write `return`, but in idiomatic Scala you almost never see it — it's mostly a debugging escape hatch.)

In [ ]:
// Simplest case — single expression body
def square(x: Int): Int = x * x

// Multi-line body — value of the block is the value of the last expression
def hypotenuse(a: Double, b: Double): Double =
  val sumOfSquares = a * a + b * b
  math.sqrt(sumOfSquares)

println(square(7))            // 49
println(hypotenuse(3.0, 4.0)) // 5.0

// A def can return Unit when it exists for its side effects
def log(level: String, msg: String): Unit =
  println(s"[$level] $msg")

log("INFO", "starting")

## The function-value form — `(x: Int) => x + 1`

The other way to introduce a function in Scala is as a **value**. The syntax `(x: Int) => x + 1` produces a function value — an object you can store in a `val`, pass as an argument, or return from another function.

A function value is a real object on the heap. Its type is one of the `FunctionN` traits in the standard library — `Function0`, `Function1`, `Function2`, all the way up to `Function22`. The number is the parameter count.

The shape:

```
(p1: T1, p2: T2, ..., pN: TN) => expression
```

For functions taking exactly one parameter, the parentheses around the parameter are optional. For functions taking zero parameters, you write `() => expression`.

In [ ]:
// Function value bound to a val
val square: Int => Int = (x: Int) => x * x

println(square(7))      // 49
println(square.apply(7))  // 49 — same thing, .apply is implicit

// Zero-arg form
val now: () => Long = () => System.currentTimeMillis()
println(now())

// Two-arg form
val add: (Int, Int) => Int = (a, b) => a + b
println(add(3, 4))

// The type ascription on the left tells the compiler what type to give
// the right-hand side. Without it, you'd write the types on the parameters:
val square2 = (x: Int) => x * x   // type inferred as Function1[Int, Int]

## Methods versus function values — the gap and the bridge

A `def` defines a **method** — a member of some enclosing class. A `(x: Int) => x + 1` introduces a **function value** — a heap object of type `Function1[Int, Int]`. On the java virtual machine these are two different things:

- A method invocation compiles to a `invokevirtual` or `invokestatic` bytecode — no object allocation.
- A function value is a real object; calling it goes through its `apply` method.

This matters because most of the standard library and most Spark ay-pee-eyes are written to take *function values* — `map`, `filter`, `foreach`. You cannot pass a `def` directly to a parameter that expects a `Function1[Int, Int]` ... except that the compiler does it for you. The conversion is called **eta-expansion** — the compiler wraps the method in a function value automatically when the context expects one.

In Scala 3, eta-expansion happens implicitly anywhere a function type is expected. In Scala 2 you had to write a trailing underscore (`square _`) to trigger it. You will see that underscore in older code; it is no longer required.

In [ ]:
// A method — not a function value
def double(x: Int): Int = x * 2

// Passing the method where a function value is expected
val xs = List(1, 2, 3, 4)
val doubled = xs.map(double)  // eta-expansion happens automatically
println(doubled)

// Explicit eta-expansion (Scala 2 style, still valid in Scala 3)
val doubleAsValue: Int => Int = double
println(doubleAsValue(10))

// The opposite direction is also free — wherever a method is expected,
// a function value works because Function1 defines apply, and Scala lets
// you call apply with method-call syntax.
println(doubleAsValue(10))     // same as doubleAsValue.apply(10)

## Higher-order functions

A *higher-order function* either takes a function as a parameter, returns a function, or both. This is the central idea that makes functional programming work — instead of writing one loop for each shape of "transform every element," you write one method that takes a transformation as an argument.

The standard library's collection methods are the canonical examples — `map`, `filter`, `foldLeft`, `reduce`, `takeWhile`. We will cover those in depth in notebook 04. Here, the shape: how you *declare* a higher-order function, and how you *call* one.

In [ ]:
// Declaring a higher-order function
// `f` is a function from Int to Int; we apply it to each element of `xs`.
def transformAll(xs: List[Int], f: Int => Int): List[Int] =
  xs.map(f)

// Calling it three different ways

// 1. Pass a previously-defined method (eta-expansion)
def triple(x: Int): Int = x * 3
println(transformAll(List(1, 2, 3), triple))      // List(3, 6, 9)

// 2. Pass an inline lambda
println(transformAll(List(1, 2, 3), x => x * 10)) // List(10, 20, 30)

// 3. Pass the underscore shorthand (next section explains it)
println(transformAll(List(1, 2, 3), _ + 100))     // List(101, 102, 103)

// Returning a function from a function
def adderFactory(base: Int): Int => Int =
  (x: Int) => base + x

val addFive = adderFactory(5)
val addTen  = adderFactory(10)

println(addFive(3))   // 8
println(addTen(3))    // 13

## Lambdas and the underscore shorthand

A lambda is just a function value written inline at the call site. The syntax variations all do the same thing — the shortest readable form is the one to use.

```scala
xs.map((x: Int) => x * 2)   // full form, with parameter type
xs.map(x => x * 2)          // type inferred from the expected parameter type of `map`
xs.map(_ * 2)               // underscore shorthand — each underscore is a fresh parameter
```

The **underscore shorthand** is the most compact form: each `_` in the expression stands for a fresh parameter, positionally. `_ + _` is `(a, b) => a + b`. It is concise but only works when each parameter is used exactly once and in the order it appears. The moment you need to reuse a parameter or use them out of order, fall back to the named-parameter form.

In [ ]:
val xs = List(1, 2, 3, 4)

// All three forms — same result
println(xs.map((x: Int) => x * 2))
println(xs.map(x => x * 2))
println(xs.map(_ * 2))

// Underscore shorthand on a two-arg function
val sum = xs.reduce(_ + _)        // (a, b) => a + b
println(sum)                      // 10

// Where underscore breaks down — using a parameter twice, or out of order
val pairsWrong = xs.map(_ -> _)   // would not compile — two _ means two params
val pairsRight = xs.map(x => x -> x)  // named — reusing x is fine
println(pairsRight)               // List((1,1), (2,2), (3,3), (4,4))

## Named arguments and default parameters

Two features that together remove the most common class of bug in any language with positional parameters: passing the right values in the wrong order.

- **Named arguments** — at the call site, you can label any argument with its parameter name. The compiler matches by name, not by position. This makes calls with many parameters readable, and lets you skip optional parameters by name.
- **Default parameters** — in the definition, any parameter can have a default value. Callers that omit it get the default.

Together they enable a style where complex configurations stay readable: `connect(host = "db.local", port = 5432, ssl = true)` is unambiguous even when the same method has eight parameters.

In [ ]:
// Default parameters
def connect(
  host: String,
  port: Int = 5432,
  user: String = "scala",
  ssl: Boolean = false,
): String =
  s"jdbc:postgresql://$host:$port/?user=$user&ssl=$ssl"

// All positional
println(connect("db.local", 5433, "ganesh", true))

// Named arguments — order doesn't matter, can skip defaults
println(connect(host = "db.local", ssl = true))

// Mix positional and named — positional must come first
println(connect("db.local", ssl = true, user = "admin"))

## Multiple parameter lists — and why they matter

A `def` can have more than one parameter list. The syntax just repeats the list:

```scala
def f(a: Int)(b: Int): Int = a + b
```

You call it the same way: `f(1)(2)`. At the surface this looks like a cosmetic variation, but it enables three things you can't get from a single parameter list.

**Type inference flows left to right.** Scala infers types one parameter list at a time. If the result of the first list constrains what the second list's lambda parameters should be, multiple lists make the inference work. You'll see this constantly in higher-order calls — `foldLeft` is declared with two lists exactly for this reason.

**Partial application.** Apply only the first list, omit the rest, and you get a function value waiting for the missing arguments. We'll see this in the section after next.

**The `using` parameter list (preview).** Scala 3 puts implicit parameters in their own parameter list with the `using` keyword. Notebook 08 covers this in depth. A method like `def foo(x: Int)(using ec: ExecutionContext)` puts the explicit parameter in the first list and the implicit one in the second.

In [ ]:
// foldLeft is the canonical example — its second parameter list takes
// the combining function, with the accumulator and element types inferred
// from the first parameter list.
val xs = List(1, 2, 3, 4)
val total = xs.foldLeft(0)((acc, x) => acc + x)
println(total)  // 10

// If foldLeft had been declared with one parameter list:
//   def foldLeft[B](z: B, op: (B, A) => B): B
// then writing `xs.foldLeft(0, (acc, x) => acc + x)` would require you
// to write the parameter types: (acc: Int, x: Int) => acc + x.
// With two lists, the seed `0` informs the inference of `B`, and the lambda
// types come for free.

// User-defined example — multiple parameter lists for the same reason
def transformWith[A, B](xs: List[A])(f: A => B): List[B] =
  xs.map(f)

// At the call site, the type of `_ * 2` is inferred from xs being List[Int]
val doubled = transformWith(List(1, 2, 3))(_ * 2)
println(doubled)

## Currying and partial application

A *curried* function is one whose parameters live in separate single-element parameter lists. So instead of `def add(a: Int, b: Int): Int`, you write `def add(a: Int)(b: Int): Int`. The two are not the same shape at the type level — the curried version has type `Int => Int => Int`, the uncurried one has type `(Int, Int) => Int`.

Why anyone bothers: a curried function makes **partial application** natural. `add(5)` gives you back a function that adds five to its argument — without writing a lambda yourself.

Scala gives you both directions:

- `.curried` on a function value returns the curried version.
- `Function.uncurried(f)` does the reverse.
- The `_` in argument position is the manual way to partially apply any method or function — fill in some arguments, leave a `_` for the rest, get back a function.

In [ ]:
// Curried form — multiple parameter lists
def addCurried(a: Int)(b: Int): Int = a + b

// Partial application — supply the first arg, get a function back
val addFive: Int => Int = addCurried(5)
println(addFive(3))   // 8
println(addFive(10))  // 15

// Uncurried form, converted to curried with .curried
val addUncurried: (Int, Int) => Int = (a, b) => a + b
val addCurried2: Int => Int => Int = addUncurried.curried
println(addCurried2(5)(3))  // 8

// Underscore partial application — works on any method
def greet(greeting: String, name: String): String =
  s"$greeting, $name"

val sayHello: String => String = greet("hello", _)
val greetGanesh: String => String = greet(_, "ganesh")

println(sayHello("ganesh"))      // hello, ganesh
println(greetGanesh("good day")) // good day, ganesh

## By-name parameters — passing the expression, not its value

By default, when you call `f(expr)`, Scala evaluates `expr` *before* calling `f`. The value goes in, not the expression.

A **by-name parameter** flips that. Declared as `x: => A` (with the `=>` arrow but no parameter list to its left), it passes the *unevaluated expression*. Each time the method body references `x`, the expression is re-evaluated.

Two real uses:

- **Control-flow ay-pee-eyes.** Building your own `if`, your own `unless`, your own retry loop. The body of the construct must not run until the right moment — by-name parameters give you that.
- **Cheap "compute only if needed."** A logging method's expensive message string should only be built if the log level is enabled. By-name lets you write `logger.debug(buildExpensiveMessage())` and have the message-building deferred.

Don't reach for by-name in everyday code. It introduces lazy evaluation in a place callers don't expect, and re-evaluating the same expression every reference can be expensive.

In [ ]:
// Eager — the expensive expression runs every time regardless
def logEager(level: String, msg: String): Unit =
  if level == "DEBUG" then println(s"[DEBUG] $msg")

// By-name — the message is only built if we actually log it
def logLazy(level: String, msg: => String): Unit =
  if level == "DEBUG" then println(s"[DEBUG] $msg")

def expensive(): String =
  println("  (computing expensive message)")
  "the message"

println("eager call with INFO level:")
logEager("INFO", expensive())   // expensive runs anyway
println("lazy call with INFO level:")
logLazy("INFO", expensive())    // expensive does not run

// Building your own control structure
def retry[A](times: Int)(body: => A): A =
  var lastError: Throwable = null
  for _ <- 1 to times do
    try return body
    catch case t: Throwable => lastError = t
  throw lastError

// Use it like a built-in
val result = retry(3) {
  println("  attempting")
  42
}
println(result)

## Vararg parameters

A `*` after a parameter type marks it as a *vararg* — the caller can pass any number of arguments of that type. Inside the method, the parameter is a `Seq` of that type.

To pass an existing collection as varargs, append `*` to the argument at the call site: `f(xs*)`. (In Scala 2 this was `xs: _*`.)

In [ ]:
def joinAll(sep: String, parts: String*): String =
  parts.mkString(sep)

println(joinAll(", ", "a", "b", "c"))        // a, b, c
println(joinAll("-", "ganesh", "scala", "3"))

// Splatting an existing collection
val tokens = List("one", "two", "three")
println(joinAll(" | ", tokens*))             // one | two | three

## The `FunctionN` family — what a function value really is

When you write `(x: Int) => x + 1`, the compiler creates an instance of `scala.Function1[Int, Int]`. `Function1` is a trait in the standard library; its only abstract method is `apply(v: T1): R`.

There is one `FunctionN` per arity from `Function0` (zero parameters) up to `Function22`. Beyond 22 you have to drop into other shapes — tuples, case classes — but you will rarely hit that limit.

This matters in three places.

- **Reading method signatures.** When the Scala docs say a parameter is of type `Int => Int`, they mean `Function1[Int, Int]`. The arrow form is sugar over the trait.
- **Defining function-like classes.** Any class that defines an `apply` method can be *called* with parentheses syntax: `myObject(arg)` desugars to `myObject.apply(arg)`. This is how `List(1, 2, 3)` works — `List` the companion object has an `apply` method.
- **Spark's own internals.** Spark's `Function1` boundary is where serialization happens — when a lambda is shipped to an executor, it is the function value that gets pickled and sent.

In [ ]:
// A lambda IS a Function1
val f: Function1[Int, Int] = (x: Int) => x + 1
println(f.apply(10))     // 11
println(f(10))           // 11 — apply called implicitly

// Any class with an apply method gets parens-call syntax
class Greeter:
  def apply(name: String): String = s"hello, $name"

val g = Greeter()
println(g("ganesh"))     // hello, ganesh — desugars to g.apply("ganesh")

// This is why List(1, 2, 3) works — List the companion has an apply
// method. List.apply(1, 2, 3) and List(1, 2, 3) are the same call.

## Methods inside methods — local functions

Methods can be defined inside other methods. The inner method is only visible within the enclosing method. Use this when you have a helper that is meaningful only in the context of one outer computation — the alternative is polluting the enclosing class with a private method nobody else should call.

Local methods close over the values in their enclosing scope, so they can reference parameters and local `val`s of the outer method directly.

In [ ]:
def formatInvoice(items: List[(String, Double)]): String =

  // Local helper — only visible inside formatInvoice
  def line(name: String, amount: Double): String =
    f"$name%-20s $$$amount%8.2f"

  // Local helper — captures `items` from the outer scope
  def total: Double =
    items.map(_._2).sum

  val body  = items.map((n, a) => line(n, a)).mkString("\n")
  val foot  = line("TOTAL", total)
  s"$body\n$foot"

println(formatInvoice(List(
  "espresso" -> 3.50,
  "croissant" -> 2.75,
  "tip" -> 1.00,
)))

## Preview — `inline def`

Scala 3 adds an `inline` keyword that turns a method into a compile-time macro: the compiler substitutes the body at the call site instead of generating a real method call.

You will rarely need this in everyday code — it is mostly relevant for library authors writing performance-critical code or compile-time computation. Mentioned here only so the keyword does not surprise you when you encounter it. Notebook 07 will use it briefly in the context of opaque types.

```scala
inline def square(x: Int): Int = x * x

// Call site:
val n = square(10)
// At compile time, becomes:
val n = 10 * 10
```

## What's next

You now have functions as first-class values, the difference between methods and function values, and the syntactic features that make Scala's function-rich style pleasant — defaults, named args, multiple parameter lists, by-name, varargs.

Notebook 04 puts these to work on the data shapes you reach for daily: `List`, `Vector`, `Set`, `Map`, `Array`, and the higher-order operations that operate over them — `map`, `filter`, `fold`, `groupBy`, and the `for-yield` sugar that desugars into exactly those calls.